# BanglishCheck: explainable detection of AI-generated Banglish

This notebook builds the whole technical project from start to finish:

1. **Dataset:** builds a length-matched set of human Banglish reviews (written before ChatGPT existed) and AI texts from Claude, ChatGPT and Gemini
2. **Classical detectors:** four baselines plus the full model (word and character TF-IDF with Banglish style features)
3. **Explanations for the classical model:** exact SHAP values for each word and style feature, plus a faithfulness test
4. **XLM-R:** a fine-tuned multilingual transformer
5. **Explanations for XLM-R:** Integrated Gradients
6. **Robustness:** a leave-one-generator-out test (is an AI model caught if the detector never saw it?)
7. **Export:** results tables, figures, and a ready-to-upload Hugging Face Space for the live demo

**Before you start:**
- Go to **Runtime > Change runtime type > T4 GPU**.
- Put these four files in Google Drive in the folder `MyDrive/banglishcheck/data/`:
  - `banglish_filtered_quality.csv`
  - `banglish_ai_samples.csv` (the Claude texts)
  - `chatgpt_banglish_samples.csv`
  - `gemini_banglish_samples.csv`

Then choose **Runtime > Run all**. It takes about 25 to 35 minutes on a T4. Everything is saved to `MyDrive/banglishcheck/outputs/`.

## 0. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/banglishcheck/data"
OUT_DIR = "/content/drive/MyDrive/banglishcheck/outputs"

FILES = {
    "human": "banglish_filtered_quality.csv",
    "claude": "banglish_ai_samples.csv",
    "chatgpt": "chatgpt_banglish_samples.csv",
    "gemini": "gemini_banglish_samples.csv",
}
AI_DOMAINS = ["ecommerce"]   # human texts are product reviews, so match the AI texts to them

XLMR_SEEDS = [42]            # use [42, 1, 2] for mean ± std over 3 runs (adds about 6 minutes)
RUN_XLMR_LOGO = True         # leave-one-generator-out for XLM-R as well (adds about 8 minutes)
SAVE_XLMR = True             # save the fine-tuned XLM-R to Drive (about 1.1 GB)

## 1. Install and set up

In [ ]:
!pip install -q "transformers>=4.46" datasets accelerate captum

from google.colab import drive
drive.mount("/content/drive")

import os, json, time, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, joblib
from IPython.display import HTML, display

OUT = Path(OUT_DIR)
for sub in ["results", "figures", "models", "space"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
for name, f in FILES.items():
    p = Path(DATA_DIR) / f
    assert p.exists(), f"Missing {p}. Put the file in Drive at that path, then run this cell again."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (XLM-R will be very slow)")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 2. Project module
This cell writes `banglishcheck.py`, which holds the features, dataset building, models, metrics and explanations. The demo app uses the same file.

In [ ]:
%%writefile banglishcheck.py
"""BanglishCheck: detect AI-generated Banglish text and explain the decision.

One module shared by the Colab notebook and the Gradio demo:
    - Banglish stylometric features
    - dataset building (human reviews + AI texts from several generators)
    - the classical detector (word + character TF-IDF + stylometric features,
      logistic regression)
    - word-level explanations (exact SHAP values for a linear model)
    - shared metrics
"""

import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
CHATGPT_RELEASE = "2022-11-30"  # human texts must be older than this

# ---------------------------------------------------------------- features
BANGLISH_WORDS = {
    "ami", "amra", "amar", "amader", "tumi", "tomar", "tomra", "apni", "apnar",
    "tar", "tader", "ki", "keno", "kivabe", "kemon", "kothay",
    "kobe", "nai", "nei", "ha", "hya", "haan", "ache", "achi", "acho",
    "chilo", "chhilo", "hobe", "hoy", "hoye", "hoyeche", "korbo", "kori", "kore",
    "korche", "korchi", "korechi", "korte", "jabo", "jai", "jacchi", "jete",
    "gelam", "gechi", "khabo", "khai", "khete", "bolo", "bolchi", "bolte",
    "dekho", "dekhi", "dekhte", "valo", "bhalo", "kharap", "onek", "ektu",
    "shob", "sob", "sobai", "eta", "ota", "oita", "eita", "ei", "oi", "ekhon",
    "ekhane", "okhane", "tokhon", "kal", "aaj", "aj", "porshu", "din", "rat",
    "raat", "bhai", "vai", "apu", "dost", "mama", "khub", "ar", "aro", "kintu",
    "tai", "jodi", "tahole", "karon", "jonno", "theke", "diye", "shathe",
    "sathe", "moto", "mone", "lagche", "lage", "parbo", "pari", "parina",
    "janina", "jani", "bujhi", "bujhlam", "thik", "accha", "achha", "acha",
    "er", "re", "ke", "te", "tho", "ta", "ti", "gula", "gulo", "dam", "taka",
    "pore", "age", "dise", "disi", "paisi", "peyechi", "nilam", "dilam",
}
ENGLISH_WORDS = {
    "the", "an", "is", "are", "was", "were", "be", "been", "am", "and",
    "but", "if", "because", "of", "in", "on", "at", "for", "with",
    "from", "by", "about", "into", "this", "that", "these", "those", "it",
    "its", "i'm", "you", "your", "he", "she", "they", "we", "our", "my",
    "him", "her", "them", "what", "which", "who", "when", "where", "why",
    "how", "not", "no", "yes", "do", "does", "did", "have", "has", "had",
    "will", "would", "can", "could", "should", "just", "very", "really",
    "too", "also", "then", "than", "there", "here", "all", "some", "more",
    "most", "good", "bad", "great", "nice", "love", "like", "time", "today",
    "product", "quality", "price", "delivery", "seller", "order", "thanks",
    "thank", "packaging", "original", "fast", "service", "size", "color",
}
AMBIGUOUS = {"to", "i", "e", "o", "se", "or", "na", "a", "so", "me"}
BANGLISH_WORDS -= AMBIGUOUS
ENGLISH_WORDS -= AMBIGUOUS | BANGLISH_WORDS

SPELLING_VARIANTS = [
    {"valo", "bhalo", "bhaalo", "vhalo", "vlo", "balo"},
    {"kemon", "kamon", "kmn"},
    {"achi", "asi", "aci"},
    {"ache", "ase", "ace", "achhe", "ase"},
    {"korbo", "krbo", "korbu"},
    {"kichu", "kisu", "kicu", "kichhu"},
    {"accha", "achha", "acha"},
    {"bhai", "vai", "bhaiya", "vaiya"},
    {"shob", "sob", "sb"},
    {"ekhon", "akhon", "ekhn"},
    {"onek", "onk", "oneek"},
    {"chilo", "cilo", "silo", "chhilo"},
]
VARIANT_WORDS = set().union(*SPELLING_VARIANTS)

WORD_RE = re.compile(r"[A-Za-z']+|[ঀ-৿]+")
SENT_RE = re.compile(r"[.!?।]+")
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF☀-➿]")
ELONGATED_RE = re.compile(r"([A-Za-z])\1{2,}")
BENGALI_RE = re.compile("[ঀ-৿]")

FEATURE_NAMES = [
    "n_words", "avg_word_len", "type_token_ratio", "banglish_word_ratio",
    "english_word_ratio", "code_switch_rate", "spelling_variant_ratio",
    "avg_sentence_len", "punctuation_rate", "repeated_punct_rate", "emoji_rate",
    "elongation_rate", "uppercase_ratio", "digit_rate",
]
FEATURE_LABELS = {
    "n_words": "Number of words",
    "avg_word_len": "Average word length",
    "type_token_ratio": "Vocabulary variety",
    "banglish_word_ratio": "Share of Bangla words",
    "english_word_ratio": "Share of English words",
    "code_switch_rate": "Bangla/English switching",
    "spelling_variant_ratio": "Informal spelling variants",
    "avg_sentence_len": "Average sentence length",
    "punctuation_rate": "Punctuation",
    "repeated_punct_rate": "Repeated punctuation (..., !!)",
    "emoji_rate": "Emoji",
    "elongation_rate": "Stretched words (sooo)",
    "uppercase_ratio": "Capital letters",
    "digit_rate": "Digits",
}


def _lang_tag(word):
    w = word.lower()
    if w in BANGLISH_WORDS:
        return "bn"
    if w in ENGLISH_WORDS:
        return "en"
    return None


def text_features(text):
    """Return a dict of FEATURE_NAMES -> float for one text."""
    text = str(text)
    words = WORD_RE.findall(text)
    lower = [w.lower() for w in words]
    n = max(len(words), 1)
    tags = [t for t in (_lang_tag(w) for w in words) if t is not None]
    switches = sum(1 for a, b in zip(tags, tags[1:]) if a != b)
    sentences = [s for s in SENT_RE.split(text) if s.strip()]
    sent_lens = [len(WORD_RE.findall(s)) for s in sentences] or [0]
    letters = [c for c in text if c.isalpha()]
    n_letters = max(len(letters), 1)
    return {
        "n_words": float(len(words)),
        "avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "type_token_ratio": len(set(lower)) / n,
        "banglish_word_ratio": sum(w in BANGLISH_WORDS for w in lower) / n,
        "english_word_ratio": sum(w in ENGLISH_WORDS for w in lower) / n,
        "code_switch_rate": switches / max(len(tags) - 1, 1),
        "spelling_variant_ratio": sum(w in VARIANT_WORDS for w in lower) / n,
        "avg_sentence_len": float(np.mean(sent_lens)),
        "punctuation_rate": sum(c in ".,!?;:-" for c in text) / max(len(text), 1),
        "repeated_punct_rate": len(re.findall(r"[.!?,]{2,}", text)) / n,
        "emoji_rate": len(EMOJI_RE.findall(text)) / n,
        "elongation_rate": len(ELONGATED_RE.findall(text)) / n,
        "uppercase_ratio": sum(c.isupper() for c in letters) / n_letters,
        "digit_rate": sum(c.isdigit() for c in text) / max(len(text), 1),
    }


class StyloFeatures(BaseEstimator, TransformerMixin):
    """scikit-learn transformer: list of texts -> (n_texts, n_features)."""

    def __init__(self, names=None):
        self.names = names

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        names = self.names or FEATURE_NAMES
        return np.array([[text_features(t)[k] for k in names] for t in X])

    def get_feature_names_out(self, input_features=None):
        return np.array(self.names or FEATURE_NAMES, dtype=object)


# ---------------------------------------------------------------- dataset
def n_words(text):
    return len(str(text).split())


def clean_text(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text.strip('"').strip()


def load_human(path, max_rows=None):
    """Human Banglish reviews written before ChatGPT's release, Latin script only."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False, nrows=max_rows)
    df["text"] = df["text"].map(clean_text)
    df = df[df["language"] == "Banglish"]
    df = df[(df["date"] != "") & (df["date"] < CHATGPT_RELEASE)]
    df = df[~df["text"].str.contains(BENGALI_RE)]
    df = df[df["text"].map(n_words).between(5, 60)]
    df = df.drop_duplicates("text")
    out = pd.DataFrame({"text": df["text"], "label": "human", "generator": "human",
                        "domain": "ecommerce", "source": "BanglishRev reviews",
                        "date": df["date"]})
    return out.reset_index(drop=True)


def load_ai(paths, domains=("ecommerce",)):
    """paths: {generator_name: csv_path}. Keeps only the given domains."""
    parts = []
    for gen, path in paths.items():
        df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        df["text"] = df["text"].map(clean_text)
        if domains:
            df = df[df["domain"].isin(domains)]
        df = df[~df["text"].str.contains(BENGALI_RE)]
        df = df[df["text"].map(n_words) >= 4]
        parts.append(pd.DataFrame({"text": df["text"], "label": "ai", "generator": gen,
                                   "domain": df["domain"], "source": gen, "date": ""}))
    out = pd.concat(parts).drop_duplicates("text")
    return out.reset_index(drop=True)


LENGTH_BINS = [0, 6, 8, 10, 12, 14, 16, 18, 20, 23, 26, 30, 40, 1000]


def length_matched_sample(human, ai, ratio=1.0, seed=SEED):
    """Sample human texts so their word-count histogram matches the AI texts.

    Without this, a model can separate the classes by length alone.
    """
    rng = np.random.default_rng(seed)
    h_bins = pd.cut(human["text"].map(n_words), LENGTH_BINS, right=False)
    a_bins = pd.cut(ai["text"].map(n_words), LENGTH_BINS, right=False)
    picks = []
    for b, want in a_bins.value_counts().items():
        pool = human.index[h_bins == b].to_numpy()
        k = min(len(pool), int(round(want * ratio)))
        if k:
            picks.extend(rng.choice(pool, size=k, replace=False))
    return human.loc[picks]


def build_dataset(human, ai, seed=SEED):
    """Length-matched, deduplicated dataset with a 70/15/15 split."""
    human = length_matched_sample(human, ai, seed=seed)
    df = pd.concat([human, ai]).drop_duplicates("text").sample(frac=1, random_state=seed)
    df = df.reset_index(drop=True)
    df.insert(0, "id", [f"{'h' if l == 'human' else 'a'}{i:05d}" for i, l in enumerate(df["label"])])
    df["y"] = (df["label"] == "ai").astype(int)
    strata = df["generator"]
    train, rest = train_test_split(df, test_size=0.30, stratify=strata, random_state=seed)
    val, test = train_test_split(rest, test_size=0.50, stratify=rest["generator"], random_state=seed)
    df["split"] = "train"
    df.loc[val.index, "split"] = "val"
    df.loc[test.index, "split"] = "test"
    return df


# ---------------------------------------------------------------- models
def make_model(kind="full", C=4.0):
    """kind: 'length', 'stylo', 'chars', 'words' or 'full'."""
    stylo = lambda names=None: Pipeline([("extract", StyloFeatures(names)),
                                         ("scale", StandardScaler())])
    words = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2,
                            sublinear_tf=True, token_pattern=r"(?u)\b\w+\b", lowercase=True)
    chars = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                            sublinear_tf=True, max_features=60000, lowercase=True)
    parts = {
        "length": [("stylo", stylo(["n_words", "avg_word_len"]))],
        "stylo": [("stylo", stylo())],
        "chars": [("chars", chars)],
        "words": [("words", words)],
        "full": [("words", words), ("chars", chars), ("stylo", stylo())],
    }[kind]
    clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
    return Pipeline([("features", FeatureUnion(parts)), ("clf", clf)])


# ---------------------------------------------------------------- metrics
def threshold_at_fpr(y_true, p_ai, max_fpr=0.05):
    """Lowest threshold whose false-positive rate is at most max_fpr."""
    fpr, _, thr = roc_curve(y_true, p_ai)
    ok = np.where(fpr <= max_fpr)[0]
    return float(thr[ok[-1]])


def evaluate(y_true, p_ai, threshold=0.5):
    y_true, p_ai = np.asarray(y_true), np.asarray(p_ai)
    y_pred = (p_ai >= threshold).astype(int)
    humans, ais = y_true == 0, y_true == 1
    res = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "human_flagged_as_ai": float(y_pred[humans].mean()) if humans.any() else None,
        "ai_caught": float(y_pred[ais].mean()) if ais.any() else None,
    }
    res["auroc"] = float(roc_auc_score(y_true, p_ai)) if len(set(y_true.tolist())) == 2 else None
    return res


# ---------------------------------------------------------------- explanations
class LinearExplainer:
    """Exact SHAP values for the 'full' logistic-regression pipeline.

    For a linear model with independent features, the SHAP value of feature j
    is coef_j * (x_j - mean_j). Word and character n-gram values are mapped
    back onto the words of the text; stylometric values are reported by name.
    """

    def __init__(self, model, background_texts):
        self.model = model
        self.union = model.named_steps["features"]
        self.coef = model.named_steps["clf"].coef_.ravel()
        X = self.union.transform(list(background_texts))
        self.mean = np.asarray(X.mean(axis=0)).ravel()
        self.names = list(self.union.get_feature_names_out())
        self.base = float(model.named_steps["clf"].intercept_[0] + self.coef @ self.mean)

    def explain(self, text):
        x = self.union.transform([text])
        x = x.tocsr() if hasattr(x, "tocsr") else x
        dense = np.asarray(x.todense()).ravel() if hasattr(x, "todense") else np.asarray(x).ravel()
        shap = self.coef * (dense - self.mean)
        words = WORD_RE.findall(text)
        lw = [w.lower() for w in words]
        padded = [f" {w} " for w in lw]
        word_scores = np.zeros(len(words))
        stylo = {}
        for j in np.flatnonzero(dense):
            name, val = self.names[j], shap[j]
            block, feat = name.split("__", 1)
            if block == "words":
                toks = feat.split()
                hits = [i for i, w in enumerate(lw) if w in toks]
                for i in hits:
                    word_scores[i] += val / len(hits)
            elif block == "chars":
                hits = [i for i, p in enumerate(padded) if feat in p]
                for i in hits:
                    word_scores[i] += val / len(hits)
        for j, name in enumerate(self.names):
            if name.startswith("stylo__"):
                stylo[name.split("__", 1)[1]] = float(shap[j])
        logit = self.base + float(shap.sum())
        return {
            "p_ai": float(1 / (1 + np.exp(-logit))),
            "words": list(zip(words, word_scores.tolist())),
            "stylo": dict(sorted(stylo.items(), key=lambda kv: -abs(kv[1]))),
        }

    def global_importance(self, top=15):
        """Largest coefficients towards AI (+) and human (-), per feature block."""
        s = pd.Series(self.coef, index=self.names)
        return s.sort_values()

    def stylo_importance(self):
        s = pd.Series({n.split("__", 1)[1]: c for n, c in zip(self.names, self.coef)
                       if n.startswith("stylo__")})
        return s.reindex(s.abs().sort_values().index)

In [ ]:
import importlib, banglishcheck as bc
importlib.reload(bc)
!cp banglishcheck.py "$OUT_DIR/space/"

## 3. Build the dataset

In [ ]:
human = bc.load_human(f"{DATA_DIR}/{FILES['human']}")
ai = bc.load_ai({g: f"{DATA_DIR}/{FILES[g]}" for g in ["claude", "chatgpt", "gemini"]}, domains=AI_DOMAINS)
print(f"Human pool (Banglish, before {bc.CHATGPT_RELEASE}, 5-60 words): {len(human):,}")
print("AI texts by generator:", ai["generator"].value_counts().to_dict())

df = bc.build_dataset(human, ai)
df.to_csv(OUT / "results" / "dataset_v1.csv", index=False)
train, val, test = (df[df.split == s] for s in ["train", "val", "test"])

summary = df.groupby(["split", "generator"]).size().unstack(fill_value=0)
display(summary)
print("Median words:", df.groupby("generator")["text"].apply(lambda s: s.map(bc.n_words).median()).to_dict())

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.arange(0, 61, 2)
for lab, color in [("human", "#2563EB"), ("ai", "#DC2626")]:
    ax.hist(df[df.label == lab]["text"].map(bc.n_words), bins=bins, alpha=.55, label=lab, color=color)
ax.set_xlabel("Words per text"); ax.set_ylabel("Texts"); ax.set_title("Length distribution after matching"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "figures" / "fig_lengths.png", dpi=200); plt.show()

## 4. Classical detectors
Every model gets the same metrics. **AI caught at 5% false flags** uses a threshold chosen on the validation set so that only 5% of human texts are flagged. It's the fairest number for a tool used on real people.

In [ ]:
results, probs = {}, {}
for kind, name in [("length", "Length only"), ("stylo", "Style features only"), ("words", "Word TF-IDF + LR"),
                   ("chars", "Char TF-IDF + LR"), ("full", "Full classical (words + chars + style)")]:
    t = time.time()
    m = bc.make_model(kind).fit(train.text, train.y)
    p_val, p_test = m.predict_proba(val.text)[:, 1], m.predict_proba(test.text)[:, 1]
    thr = bc.threshold_at_fpr(val.y, p_val)
    r = bc.evaluate(test.y, p_test)
    r["ai_caught_at_5pct_false_flags"] = bc.evaluate(test.y, p_test, thr)["ai_caught"]
    r["train_seconds"] = round(time.time() - t, 1)
    results[name], probs[name] = r, p_test
    if kind == "full":
        full_model, full_thr = m, thr
table = pd.DataFrame(results).T[["accuracy", "macro_f1", "auroc", "human_flagged_as_ai", "ai_caught_at_5pct_false_flags", "train_seconds"]]
display(table.round(3))

## 5. Explaining the classical model (exact SHAP)
For a linear model, the SHAP value of each feature is `coefficient × (value − average value)`. The values below are mapped back onto the words of each text.

In [ ]:
explainer = bc.LinearExplainer(full_model, train.text.sample(min(1000, len(train)), random_state=bc.SEED))
joblib.dump(explainer, OUT / "space" / "explainer.joblib")
joblib.dump(full_model, OUT / "models" / "classical_full.joblib")

# Global view: which style features push towards AI (+) or human (-)
s = explainer.stylo_importance().rename(index=bc.FEATURE_LABELS)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(s.index, s.values, color=["#DC2626" if v > 0 else "#2563EB" for v in s.values])
ax.axvline(0, color="k", lw=.7); ax.set_title("Style features: weight towards AI (+) or human (−)")
fig.tight_layout(); fig.savefig(OUT / "figures" / "fig_style_features.png", dpi=200); plt.show()

g = explainer.global_importance()
top = pd.concat([g.head(12), g.tail(12)])
top.index = [n.replace("words__", "w: ").replace("chars__", "c: ").replace("stylo__", "s: ") for n in top.index]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top.index, top.values, color=["#DC2626" if v > 0 else "#2563EB" for v in top.values])
ax.axvline(0, color="k", lw=.7); ax.set_title("Strongest features overall (w = word, c = character n-gram)")
fig.tight_layout(); fig.savefig(OUT / "figures" / "fig_top_features.png", dpi=200); plt.show()

In [ ]:
def render(pairs, title):
    """pairs: list of (word, score). Red = towards AI, blue = towards human."""
    m = max([abs(s) for _, s in pairs] + [1e-9])
    spans = []
    for w, sc in pairs:
        a = min(abs(sc) / m, 1) * .75
        col = f"rgba(220,38,38,{a:.2f})" if sc > 0 else f"rgba(37,99,235,{a:.2f})"
        spans.append(f"<span style='background:{col};padding:1px 3px;border-radius:3px'>{w}</span>")
    return f"<div style='margin:8px 0'><b>{title}</b><br>{' '.join(spans)}</div>"

rng = np.random.default_rng(0)
pred_full = (full_model.predict_proba(test.text)[:, 1] >= .5).astype(int)
picks = list(test[(test.y == 0) & (pred_full == 0)].index[:2]) + list(test[(test.y == 1) & (pred_full == 1)].index[:2]) \
      + list(test[pred_full != test.y].index[:2])
html = ""
for i in picks:
    e = explainer.explain(test.loc[i, "text"])
    title = f"True: {test.loc[i, 'generator']} · P(AI) = {e['p_ai']:.2f}"
    html += render(e["words"], title)
display(HTML(html))
(OUT / "results" / "classical_examples.html").write_text(html, encoding="utf-8")

In [ ]:
# Faithfulness: removing the 5 most important words should change the prediction
# much more than removing 5 random words.
def drop_words(text, idx):
    words = bc.WORD_RE.findall(text)
    return " ".join(w for j, w in enumerate(words) if j not in idx)

rows = []
for text in test[test.text.map(bc.n_words) >= 10].text.sample(200, random_state=1):
    e = explainer.explain(text)
    p0 = e["p_ai"]; sign = 1 if p0 >= .5 else -1
    scores = np.array([s for _, s in e["words"]]) * sign
    top5 = set(np.argsort(-scores)[:5]); rand5 = set(rng.choice(len(scores), min(5, len(scores)), replace=False))
    f = lambda t: full_model.predict_proba([t])[0, 1]
    rows.append({"top5_drop": sign * (p0 - f(drop_words(text, top5))), "random5_drop": sign * (p0 - f(drop_words(text, rand5)))})
faith_classical = pd.DataFrame(rows).mean()
print("Average confidence drop when removing 5 words (classical model):"); print(faith_classical.round(3))

## 6. Fine-tune XLM-R

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)
MODEL_NAME, MAX_LEN = "xlm-roberta-base", 128
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(part):
    d = Dataset.from_pandas(part[["text", "y"]].rename(columns={"y": "labels"}).reset_index(drop=True))
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])

def softmax_ai(logits):
    return torch.softmax(torch.tensor(logits, dtype=torch.float32), -1)[:, 1].numpy()

def compute_metrics(pred):
    r = bc.evaluate(pred.label_ids, softmax_ai(pred.predictions))
    return {k: r[k] for k in ["accuracy", "macro_f1", "auroc"]}

def train_xlmr(tr, va, seed, out_dir):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = TrainingArguments(
        output_dir=out_dir, eval_strategy="epoch", save_strategy="epoch", learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=64, num_train_epochs=3,
        warmup_ratio=0.1, weight_decay=0.01, fp16=torch.cuda.is_available(),
        load_best_model_at_end=True, metric_for_best_model="macro_f1", save_total_limit=1,
        logging_steps=25, report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=to_ds(tr), eval_dataset=to_ds(va),
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    trainer.train()
    return trainer

def xlmr_proba(trainer, part):
    return softmax_ai(trainer.predict(to_ds(part)).predictions)

runs = []
for seed in XLMR_SEEDS:
    t = time.time()
    trainer = train_xlmr(train, val, seed, f"/content/xlmr_seed{seed}")
    p_val, p_test = xlmr_proba(trainer, val), xlmr_proba(trainer, test)
    r = bc.evaluate(test.y, p_test)
    r["ai_caught_at_5pct_false_flags"] = bc.evaluate(test.y, p_test, bc.threshold_at_fpr(val.y, p_val))["ai_caught"]
    r["train_seconds"] = round(time.time() - t, 1)
    runs.append(r)
    if seed == XLMR_SEEDS[0]:
        xlmr, probs["XLM-R (fine-tuned)"] = trainer, p_test
runs = pd.DataFrame(runs)
results["XLM-R (fine-tuned)"] = runs.mean().to_dict()
if len(runs) > 1:
    print("XLM-R over seeds (mean ± std):")
    display(pd.DataFrame({"mean": runs.mean(), "std": runs.std()}).round(3))
if SAVE_XLMR:
    xlmr.save_model(str(OUT / "models" / "xlmr")); tok.save_pretrained(str(OUT / "models" / "xlmr"))
table = pd.DataFrame(results).T[["accuracy", "macro_f1", "auroc", "human_flagged_as_ai", "ai_caught_at_5pct_false_flags", "train_seconds"]]
display(table.round(3))

## 7. Explaining XLM-R (Integrated Gradients)

In [ ]:
from captum.attr import LayerIntegratedGradients
model = xlmr.model.eval().float()
dev = model.device

def ai_prob(input_ids, attention_mask):
    return torch.softmax(model(input_ids=input_ids, attention_mask=attention_mask).logits, -1)[:, 1]

lig = LayerIntegratedGradients(ai_prob, model.roberta.embeddings)

def ig_words(text, n_steps=50):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    ids, mask = enc["input_ids"].to(dev), enc["attention_mask"].to(dev)
    ref = torch.full_like(ids, tok.pad_token_id); ref[0, 0], ref[0, -1] = ids[0, 0], ids[0, -1]
    attr, delta = lig.attribute(ids, ref, additional_forward_args=(mask,), n_steps=n_steps,
                                internal_batch_size=25, return_convergence_delta=True)
    tok_scores = attr.sum(-1).squeeze(0).detach().cpu().numpy()
    words = {}
    for i, w in enumerate(enc.word_ids(0)):
        if w is not None:
            words[w] = words.get(w, 0.0) + float(tok_scores[i])
    pairs = []
    for w, sc in sorted(words.items()):
        span = enc.word_to_chars(0, w)
        pairs.append((text[span.start:span.end], sc))
    with torch.no_grad():
        p = float(ai_prob(ids, mask))
    return pairs, p, float(delta)

html = ""
for i in picks:
    pairs, p, delta = ig_words(test.loc[i, "text"])
    html += render(pairs, f"True: {test.loc[i, 'generator']} · XLM-R P(AI) = {p:.2f} · IG delta = {delta:.3f}")
display(HTML(html))
(OUT / "results" / "xlmr_ig_examples.html").write_text(html, encoding="utf-8")

# Same faithfulness test for IG
def xlmr_p(texts):
    enc = tok(list(texts), return_tensors="pt", truncation=True, max_length=MAX_LEN, padding=True).to(dev)
    with torch.no_grad():
        return ai_prob(enc["input_ids"], enc["attention_mask"]).cpu().numpy()

rows = []
for text in test[test.text.map(bc.n_words) >= 10].text.sample(100, random_state=1):
    pairs, p0, _ = ig_words(text, n_steps=30)
    sign = 1 if p0 >= .5 else -1
    scores = np.array([s for _, s in pairs]) * sign
    words = [w for w, _ in pairs]
    top5 = set(np.argsort(-scores)[:5]); rand5 = set(rng.choice(len(words), 5, replace=False))
    keep = lambda idx: " ".join(w for j, w in enumerate(words) if j not in idx)
    p_top, p_rand = xlmr_p([keep(top5), keep(rand5)])
    rows.append({"top5_drop": sign * (p0 - p_top), "random5_drop": sign * (p0 - p_rand)})
faith_xlmr = pd.DataFrame(rows).mean()
print("Average confidence drop when removing 5 words (XLM-R + IG):"); print(faith_xlmr.round(3))
pd.DataFrame({"Classical + SHAP": faith_classical, "XLM-R + IG": faith_xlmr}).to_csv(OUT / "results" / "faithfulness.csv")

## 8. Robustness: leave one generator out
Train without one AI model's texts, then test on that model's texts plus the human test set. This shows whether the detector would catch a model it has never seen.

In [ ]:
logo = []
for g in ["claude", "chatgpt", "gemini"]:
    tr, va = train[train.generator != g], val[val.generator != g]
    te = pd.concat([test[test.y == 0], df[df.generator == g]])
    m = bc.make_model("full").fit(tr.text, tr.y)
    pv, pt = m.predict_proba(va.text)[:, 1], m.predict_proba(te.text)[:, 1]
    thr = bc.threshold_at_fpr(va.y, pv)
    logo.append({"held_out": g, "model": "Full classical", "auroc": bc.evaluate(te.y, pt)["auroc"],
                 "ai_caught_at_5pct_false_flags": bc.evaluate(te.y, pt, thr)["ai_caught"]})
    if RUN_XLMR_LOGO:
        trn = train_xlmr(tr, va, XLMR_SEEDS[0], f"/content/xlmr_logo_{g}")
        pv, pt = xlmr_proba(trn, va), xlmr_proba(trn, te)
        thr = bc.threshold_at_fpr(va.y, pv)
        logo.append({"held_out": g, "model": "XLM-R", "auroc": bc.evaluate(te.y, pt)["auroc"],
                     "ai_caught_at_5pct_false_flags": bc.evaluate(te.y, pt, thr)["ai_caught"]})
        del trn; torch.cuda.empty_cache()
logo = pd.DataFrame(logo)
logo.to_csv(OUT / "results" / "leave_one_generator_out.csv", index=False)
display(logo.round(3))

piv = logo.pivot(index="held_out", columns="model", values="ai_caught_at_5pct_false_flags")
ax = piv.plot.bar(rot=0, figsize=(7, 3.8), ylim=(0, 1), color=["#2563EB", "#DC2626"][:piv.shape[1]])
ax.set_ylabel("AI texts caught (5% false flags)"); ax.set_xlabel("Generator never seen in training")
ax.set_title("Detecting an unseen AI model"); plt.tight_layout()
plt.savefig(OUT / "figures" / "fig_unseen_generator.png", dpi=200); plt.show()

## 9. Final results and figures

In [ ]:
from sklearn.metrics import roc_curve
table = pd.DataFrame(results).T[["accuracy", "macro_f1", "auroc", "human_flagged_as_ai", "ai_caught_at_5pct_false_flags", "train_seconds"]]
table.to_csv(OUT / "results" / "model_comparison.csv")
display(table.round(3))
try:
    print(table.round(3).to_markdown())
except ImportError:
    pass

fig, ax = plt.subplots(figsize=(5.5, 5))
for name in ["Length only", "Style features only", "Full classical (words + chars + style)", "XLM-R (fine-tuned)"]:
    fpr, tpr, _ = roc_curve(test.y, probs[name])
    ax.plot(fpr, tpr, label=f"{name} (AUC {results[name]['auroc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=.7); ax.set_xlabel("Human texts flagged as AI"); ax.set_ylabel("AI texts caught")
ax.set_title("ROC curves on the test set"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figures" / "fig_roc.png", dpi=200); plt.show()

ax = table[["macro_f1", "ai_caught_at_5pct_false_flags"]].plot.barh(figsize=(8, 4), xlim=(0, 1))
ax.set_title("Model comparison (test set)"); plt.tight_layout()
plt.savefig(OUT / "figures" / "fig_models.png", dpi=200); plt.show()

per_gen = []
for name in ["Full classical (words + chars + style)", "XLM-R (fine-tuned)"]:
    pred = (probs[name] >= .5).astype(int)
    for g in ["claude", "chatgpt", "gemini"]:
        mask = (test.generator == g).values
        per_gen.append({"model": name, "generator": g, "ai_caught": pred[mask].mean()})
per_gen = pd.DataFrame(per_gen).pivot(index="generator", columns="model", values="ai_caught")
per_gen.to_csv(OUT / "results" / "caught_by_generator.csv"); display(per_gen.round(3))

## 10. Build the live demo (Hugging Face Space)
This writes the demo app next to the saved model and zips it. The next cell checks the app works here before you upload it.

In [ ]:
%%writefile app.py
"""BanglishCheck demo: paste Banglish text, get an AI score and the reasons.

Runs on a free Hugging Face Space (CPU). Needs, next to this file:
    banglishcheck.py   the shared module
    explainer.joblib   saved by the Colab notebook
"""

import gradio as gr
import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from banglishcheck import FEATURE_LABELS, WORD_RE, n_words

explainer = joblib.load("explainer.joblib")

EXAMPLES = [
    "onek valo product. packaging o onek valo chilo. seller vai ke dhonnobad 🥰",
    "Daraz theke earbuds order dilam 3 din age, tracking e dekhay 'out for delivery' kintu rider er kono khobor nai.",
    "Product er quality dekhe mone hocche price ta worth it, packaging o neat and clean chilo.",
    "size ta aktu choto hoise.. but overall kharap na,, nite paren",
]

NOTICE = (
    "**Please read:** this score is an estimate from a student project. It can be wrong, "
    "especially on short texts and on writing styles it hasn't seen. Never use it to accuse "
    "anyone of using AI."
)


def stylo_plot(stylo, top=8):
    items = list(stylo.items())[:top][::-1]
    labels = [FEATURE_LABELS.get(k, k) for k, _ in items]
    vals = [v for _, v in items]
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.barh(labels, vals, color=["#C2410C" if v > 0 else "#0F766E" for v in vals])
    ax.axvline(0, color="#444", lw=0.8)
    ax.set_xlabel("pushes towards Human (−)   |   towards AI (+)")
    ax.set_title("Writing-style signals")
    fig.tight_layout()
    return fig


def analyse(text):
    text = (text or "").strip()
    if not text:
        return {}, [], None, "Paste some Banglish text first."
    res = explainer.explain(text)
    p = res["p_ai"]
    biggest = max((abs(s) for _, s in res["words"]), default=1.0) or 1.0
    # Rebuild the full text: scored words, with the spaces, punctuation and
    # emoji between them kept unscored.
    highlighted, pos = [], 0
    for m, (_, score) in zip(WORD_RE.finditer(text), res["words"]):
        if m.start() > pos:
            highlighted.append((text[pos:m.start()], None))
        highlighted.append((m.group(), round(score / biggest, 3)))
        pos = m.end()
    if pos < len(text):
        highlighted.append((text[pos:], None))
    note = NOTICE
    if n_words(text) < 8:
        note = "⚠️ This text is very short, so the score is unreliable.\n\n" + NOTICE
    return {"AI-generated": p, "Human-written": 1 - p}, highlighted, stylo_plot(res["stylo"]), note


with gr.Blocks(title="BanglishCheck") as demo:
    gr.Markdown(
        "# BanglishCheck\n"
        "Paste a Banglish text (Bangla written in English letters). The detector estimates whether "
        "it was written by a person or by an AI model, and shows which words and writing-style "
        "signals drove the decision."
    )
    with gr.Row():
        with gr.Column():
            inp = gr.Textbox(label="Banglish text", lines=5, placeholder="ami kal class e jabo na, too tired...")
            btn = gr.Button("Check", variant="primary")
            gr.Examples(EXAMPLES, inputs=inp)
        with gr.Column():
            label = gr.Label(label="Prediction", num_top_classes=2)
            words = gr.HighlightedText(label="Word evidence: red = AI-like, blue = human-like")
            plot = gr.Plot(label="Writing-style signals")
            note = gr.Markdown(NOTICE)
    btn.click(analyse, inputs=inp, outputs=[label, words, plot, note])
    inp.submit(analyse, inputs=inp, outputs=[label, words, plot, note])

if __name__ == "__main__":
    demo.launch()

In [ ]:
import sklearn, matplotlib
!cp app.py "$OUT_DIR/space/"
req = f"""gradio==6.29.1
scikit-learn=={sklearn.__version__}
numpy=={np.__version__}
pandas=={pd.__version__}
matplotlib=={matplotlib.__version__}
joblib=={joblib.__version__}
"""
(OUT / "space" / "requirements.txt").write_text(req)
(OUT / "space" / "README.md").write_text("""---
title: BanglishCheck
emoji: 🔍
colorFrom: blue
colorTo: red
sdk: gradio
sdk_version: 6.29.1
app_file: app.py
pinned: false
---

# BanglishCheck
Paste Banglish text to estimate whether it was written by a person or an AI model, with word-level and
writing-style explanations. Part of a Research Den internship project.
""")
!cd "$OUT_DIR" && rm -f space.zip && zip -qr space.zip space
print(req)
print("Space files:", sorted(os.listdir(OUT / "space")))

In [ ]:
# Test the demo inside Colab. A public link appears below for about 72 hours.
!pip install -q gradio==6.29.1
import importlib, shutil
os.chdir(OUT / "space")
import app as demo_app
importlib.reload(demo_app)
demo_app.demo.launch(share=True)